# Artificial Intelligence Nanodegree
## Machine Translation Project
In this notebook, sections that end with **'(IMPLEMENTATION)'** in the header indicate that the following blocks of code will require additional functionality which you must provide. Please be sure to read the instructions carefully!

## Introduction
In this notebook, you will build a deep neural network that functions as part of an end-to-end machine translation pipeline. Your completed pipeline will accept English text as input and return the French translation.

- **Preprocess** - You'll convert text to sequence of integers.
- **Models** Create models which accepts a sequence of integers as input and returns a probability distribution over possible translations. After learning about the basic types of neural networks that are often used for machine translation, you will engage in your own investigations, to design your own model!
- **Prediction** Run the model on English text.

In [1]:
%load_ext autoreload
%aimport helper, project_tests
%autoreload 1

2025-12-06 06:00:54.153157: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-06 06:00:54.153225: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-06 06:00:54.153290: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-06 06:00:54.166984: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Should return keras==2.14.0 and tf-keras==2.14.1
!pip freeze | grep keras

keras==2.14.0
tf-keras==2.14.1


In [3]:
# Should return tensorflow==2.14.0
!pip freeze | grep tensorflow

tensorflow==2.14.0
tensorflow-datasets==4.9.4
tensorflow-estimator==2.14.0
tensorflow-hub==0.16.1
tensorflow-io-gcs-filesystem==0.34.0
tensorflow-metadata==1.15.0


In [4]:
import collections

import helper
import numpy as np
import project_tests as tests

from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from tensorflow.keras.layers import GRU, Input, Dense, TimeDistributed, Activation, RepeatVector, Bidirectional
from keras.layers import Embedding
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy

## Dataset
We begin by investigating the dataset that will be used to train and evaluate your pipeline.  The most common datasets used for machine translation are from [WMT](http://www.statmt.org/).  However, that will take a long time to train a neural network on.  We'll be using a dataset we created for this project that contains a small vocabulary.  You'll be able to train your model in a reasonable time with this dataset.
### Load Data
The data is located in `data/small_vocab_en` and `data/small_vocab_fr`. The `small_vocab_en` file contains English sentences with their French translations in the `small_vocab_fr` file. Load the English and French data from these files from running the cell below.

In [5]:
# Load English data
english_sentences = helper.load_data('data/small_vocab_en')
# Load French data
french_sentences = helper.load_data('data/small_vocab_fr')

print('Dataset Loaded')

Dataset Loaded


### Files
Each line in `small_vocab_en` contains an English sentence with the respective translation in each line of `small_vocab_fr`.  View the first two lines from each file.

In [6]:
for sample_i in range(2):
    print('small_vocab_en Line {}:  {}'.format(sample_i + 1, english_sentences[sample_i]))
    print('small_vocab_fr Line {}:  {}'.format(sample_i + 1, french_sentences[sample_i]))

small_vocab_en Line 1:  new jersey is sometimes quiet during autumn , and it is snowy in april .
small_vocab_fr Line 1:  new jersey est parfois calme pendant l' automne , et il est neigeux en avril .
small_vocab_en Line 2:  the united states is usually chilly during july , and it is usually freezing in november .
small_vocab_fr Line 2:  les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .


From looking at the sentences, you can see they have been preprocessed already.  The puncuations have been delimited using spaces. All the text have been converted to lowercase.  This should save you some time, but the text requires more preprocessing.
### Vocabulary
The complexity of the problem is determined by the complexity of the vocabulary.  A more complex vocabulary is a more complex problem.  Let's look at the complexity of the dataset we'll be working with.

In [7]:
english_words_counter = collections.Counter([word for sentence in english_sentences for word in sentence.split()])
french_words_counter = collections.Counter([word for sentence in french_sentences for word in sentence.split()])

print('{} English words.'.format(len([word for sentence in english_sentences for word in sentence.split()])))
print('{} unique English words.'.format(len(english_words_counter)))
print('10 Most common words in the English dataset:')
print('"' + '" "'.join(list(zip(*english_words_counter.most_common(10)))[0]) + '"')
print()
print('{} French words.'.format(len([word for sentence in french_sentences for word in sentence.split()])))
print('{} unique French words.'.format(len(french_words_counter)))
print('10 Most common words in the French dataset:')
print('"' + '" "'.join(list(zip(*french_words_counter.most_common(10)))[0]) + '"')

1823250 English words.
227 unique English words.
10 Most common words in the English dataset:
"is" "," "." "in" "it" "during" "the" "but" "and" "sometimes"

1961295 French words.
355 unique French words.
10 Most common words in the French dataset:
"est" "." "," "en" "il" "les" "mais" "et" "la" "parfois"


For comparison, _Alice's Adventures in Wonderland_ contains 2,766 unique words of a total of 15,500 words.
## Preprocess
For this project, you won't use text data as input to your model. Instead, you'll convert the text into sequences of integers using the following preprocess methods:
1. Tokenize the words into ids
2. Add padding to make all the sequences the same length.

Time to start preprocessing the data...
### Tokenize (IMPLEMENTATION)
For a neural network to predict on text data, it first has to be turned into data it can understand. Text data like "dog" is a sequence of ASCII character encodings.  Since a neural network is a series of multiplication and addition operations, the input data needs to be number(s).

We can turn each character into a number or each word into a number.  These are called character and word ids, respectively.  Character ids are used for character level models that generate text predictions for each character.  A word level model uses word ids that generate text predictions for each word.  Word level models tend to learn better, since they are lower in complexity, so we'll use those.

Turn each sentence into a sequence of words ids using Keras's [`Tokenizer`](https://keras.io/preprocessing/text/#tokenizer) function. Use this function to tokenize `english_sentences` and `french_sentences` in the cell below.

Running the cell will run `tokenize` on sample data and show output for debugging.

In [10]:
def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    # TODO: Implement
    from tensorflow.keras.preprocessing.text import Tokenizer

def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(x)
    x_tokenized = tokenizer.texts_to_sequences(x)
    return x_tokenized, tokenizer

# Tokenize Example output
text_sentences = [
    'The quick brown fox jumps over the lazy dog .',
    'By Jove , my quick study of lexicography won a prize .',
    'This is a short sentence .']
text_tokenized, text_tokenizer = tokenize(text_sentences)
print(text_tokenizer.word_index)
print()
for sample_i, (sent, token_sent) in enumerate(zip(text_sentences, text_tokenized)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(sent))
    print('  Output: {}'.format(token_sent))

{'the': 1, 'quick': 2, 'a': 3, 'brown': 4, 'fox': 5, 'jumps': 6, 'over': 7, 'lazy': 8, 'dog': 9, 'by': 10, 'jove': 11, 'my': 12, 'study': 13, 'of': 14, 'lexicography': 15, 'won': 16, 'prize': 17, 'this': 18, 'is': 19, 'short': 20, 'sentence': 21}

Sequence 1 in x
  Input:  The quick brown fox jumps over the lazy dog .
  Output: [1, 2, 4, 5, 6, 7, 1, 8, 9]
Sequence 2 in x
  Input:  By Jove , my quick study of lexicography won a prize .
  Output: [10, 11, 12, 2, 13, 14, 15, 16, 3, 17]
Sequence 3 in x
  Input:  This is a short sentence .
  Output: [18, 19, 3, 20, 21]


### Padding (IMPLEMENTATION)
When batching the sequence of word ids together, each sequence needs to be the same length.  Since sentences are dynamic in length, we can add padding to the end of the sequences to make them the same length.

Make sure all the English sequences have the same length and all the French sequences have the same length by adding padding to the **end** of each sequence using Keras's [`pad_sequences`](https://keras.io/preprocessing/sequence/#pad_sequences) function.

In [17]:
import numpy as np

def pad(x, length=None):
    """
    Pad x
    :param x: List of sequences.
    :param length: Length to pad the sequence to. If None, use length of longest sequence in x.
    :return: Padded numpy array of sequences
    """
    if length is None:
        length = max(len(seq) for seq in x)

    # Create padded matrix
    padded_x = np.zeros((len(x), length), dtype=int)

    for i, seq in enumerate(x):
        padded_x[i, :len(seq)] = seq

    return padded_x

test_pad = pad(text_tokenized)
for sample_i, (token_sent, pad_sent) in enumerate(zip(text_tokenized, test_pad)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))


Sequence 1 in x
  Input:  [1 2 4 5 6 7 1 8 9]
  Output: [1 2 4 5 6 7 1 8 9 0]
Sequence 2 in x
  Input:  [10 11 12  2 13 14 15 16  3 17]
  Output: [10 11 12  2 13 14 15 16  3 17]
Sequence 3 in x
  Input:  [18 19  3 20 21]
  Output: [18 19  3 20 21  0  0  0  0  0]


### Preprocess Pipeline
Your focus for this project is to build neural network architecture, so we won't ask you to create a preprocess pipeline.  Instead, we've provided you with the implementation of the `preprocess` function.

In [14]:
def preprocess(x, y):
    """
    Preprocess x and y
    :param x: Feature List of sentences
    :param y: Label List of sentences
    :return: Tuple of (Preprocessed x, Preprocessed y, x tokenizer, y tokenizer)
    """
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    # Keras's sparse_categorical_crossentropy function requires the labels to be in 3 dimensions
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer =\
    preprocess(english_sentences, french_sentences)
    
max_english_sequence_length = preproc_english_sentences.shape[1]
max_french_sequence_length = preproc_french_sentences.shape[1]
english_vocab_size = len(english_tokenizer.word_index)
french_vocab_size = len(french_tokenizer.word_index)

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max French sentence length:", max_french_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

Data Preprocessed
Max English sentence length: 15
Max French sentence length: 21
English vocabulary size: 199
French vocabulary size: 344


## Models
In this section, you will experiment with various neural network architectures.
You will begin by training four relatively simple architectures.
- Model 1 is a simple RNN
- Model 2 is a RNN with Embedding
- Model 3 is a Bidirectional RNN
- Model 4 is an Encoder-Decoder RNN

After experimenting with the four simple architectures, you will construct a deeper architecture that is designed to outperform all four models.
### Ids Back to Text
The neural network will be translating the input to words ids, which isn't the final form we want.  We want the French translation.  The function `logits_to_text` will bridge the gab between the logits from the neural network to the French translation.  You'll be using this function to better understand the output of the neural network.

In [15]:
def logits_to_text(logits, tokenizer):
    """
    Turn logits from a neural network into text using the tokenizer
    :param logits: Logits from a neural network
    :param tokenizer: Keras Tokenizer fit on the labels
    :return: String that represents the text of the logits
    """
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

print('`logits_to_text` function loaded.')

`logits_to_text` function loaded.


### Model 1: RNN (IMPLEMENTATION)
![RNN](images/rnn.png)
A basic RNN model is a good baseline for sequence data.  In this model, you'll build a RNN that translates English to French.

In [20]:
from tensorflow.keras.layers import Input, GRU, TimeDistributed, Dense
from tensorflow.keras.models import Model

learning_rate = 0.001

def simple_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a basic RNN on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # Input shape: (batch_size, timesteps, features)
    input_seq = Input(shape=input_shape[1:])

    # Simple GRU layer that returns a sequence (one output per time step)
    rnn = GRU(64, return_sequences=True)(input_seq)

    # TimeDistributed Dense layer to get a probability distribution over
    # the French vocabulary for each time step
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(rnn)

    # Build the model
    model = Model(inputs=input_seq, outputs=logits)

    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model


tests.test_simple_model(simple_model)

# Reshaping the input to work with a basic RNN
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

# Train the neural network
simple_rnn_model = simple_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)
simple_rnn_model.fit(tmp_x, preproc_french_sentences, batch_size=1024, epochs=10, validation_split=0.2)

# Print prediction(s)
print(logits_to_text(simple_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10


2025-12-06 06:18:54.197393: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8600
2025-12-06 06:18:56.179697: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7e970232bb80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-06 06:18:56.179747: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
2025-12-06 06:18:56.185942: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-06 06:18:56.308490: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


108/108 [==============================] - 6s 19ms/step - loss: 3.4708 - accuracy: 0.4247 - val_loss: nan - val_accuracy: 0.4691
Epoch 2/10
108/108 [==============================] - 1s 14ms/step - loss: 2.4087 - accuracy: 0.4734 - val_loss: nan - val_accuracy: 0.4822
Epoch 3/10
108/108 [==============================] - 1s 14ms/step - loss: 2.1517 - accuracy: 0.5211 - val_loss: nan - val_accuracy: 0.5490
Epoch 4/10
108/108 [==============================] - 2s 14ms/step - loss: 1.9298 - accuracy: 0.5591 - val_loss: nan - val_accuracy: 0.5725
Epoch 5/10
108/108 [==============================] - 1s 14ms/step - loss: 1.7700 - accuracy: 0.5743 - val_loss: nan - val_accuracy: 0.5778
Epoch 6/10
108/108 [==============================] - 2s 14ms/step - loss: 1.6644 - accuracy: 0.5809 - val_loss: nan - val_accuracy: 0.5862
Epoch 7/10
108/108 [==============================] - 2s 14ms/step - loss: 1.5896 - accuracy: 0.5881 - val_loss: nan - val_accuracy: 0.5928
Epoch 8/10
108/108 [===========

We used a **GRU layer with 64 units** because GRUs are computationally efficient and easier to train than LSTMs, while still being capable of capturing short-term temporal dependencies. The number of units was kept small to ensure fast training and to prevent overfitting on this relatively small dataset. The model also used a **TimeDistributed Dense layer with a softmax activation**, which produces a probability distribution over all French vocabulary words at each predicted time step.

A **learning rate of 0.001** was chosen because it is a commonly effective default for the Adam optimizer and provides stable convergence without large oscillations. Zero-padding was used to ensure all sequences had equal length, which is required for feeding batches into recurrent networks.

Although the simple model learned some structure and produced partially correct translations, its performance is limited. Without word embeddings, an encoder–decoder structure, or attention mechanisms, the model struggles with long-range dependencies and tends to repeat high-frequency words. This is expected and highlights the need for more advanced architectures.

In the following models, we can improve performance by adding embedding layers, deeper recurrent networks, and dedicated encoder–decoder structures that better capture sentence meaning and context.

### Model 2: Embedding (IMPLEMENTATION)
![RNN](images/embedding.png)
You've turned the words into ids, but there's a better representation of a word.  This is called word embeddings.  An embedding is a vector representation of the word that is close to similar words in n-dimensional space, where the n represents the size of the embedding vectors.

In this model, you'll create a RNN model using embedding.

In [21]:
def embed_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a RNN model using word embedding on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    from tensorflow.keras.layers import Input, Embedding, GRU, TimeDistributed, Dense
    from tensorflow.keras.models import Model

    # Input: sequences of word indices (no extra feature dimension)
    input_seq = Input(shape=(input_shape[1],))

    # Embedding layer to learn dense word representations
    embed = Embedding(input_dim=english_vocab_size,
                      output_dim=64,
                      input_length=output_sequence_length)(input_seq)

    # Recurrent layer that outputs a sequence (one vector per time step)
    rnn = GRU(64, return_sequences=True)(embed)

    # TimeDistributed Dense over the French vocabulary with softmax
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(rnn)

    # Build functional model
    model = Model(inputs=input_seq, outputs=logits)

    # Compile – reuse the same learning_rate as before (defined earlier)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model


tests.test_embed_model(embed_model)


# TODO: Reshape the input
# For the embedding model, we keep integer sequences with shape (batch, timesteps)
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)

# TODO: Train the neural network
embed_rnn_model = embed_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

embed_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

# TODO: Print prediction(s)
print(logits_to_text(embed_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 7s 45ms/step - loss: 3.7877 - accuracy: 0.3941 - val_loss: nan - val_accuracy: 0.4093
Epoch 2/10
108/108 [==============================] - 2s 22ms/step - loss: 2.6580 - accuracy: 0.4493 - val_loss: nan - val_accuracy: 0.4962
Epoch 3/10
108/108 [==============================] - 2s 15ms/step - loss: 2.0214 - accuracy: 0.5516 - val_loss: nan - val_accuracy: 0.6039
Epoch 4/10
108/108 [==============================] - 2s 16ms/step - loss: 1.4657 - accuracy: 0.6423 - val_loss: nan - val_accuracy: 0.6953
Epoch 5/10
108/108 [==============================] - 2s 17ms/step - loss: 1.1100 - accuracy: 0.7285 - val_loss: nan - val_accuracy: 0.7540
Epoch 6/10
108/108 [==============================] - 2s 16ms/step - loss: 0.9035 - accuracy: 0.7680 - val_loss: nan - val_accuracy: 0.7804
Epoch 7/10
108/108 [==============================] - 2s 15ms/step - loss: 0.7705 - accuracy: 0.7941 - val_loss: nan - val_accuracy: 0.8065
Epoch 8/10
108/108 [

## **Conclusion for the Embedding Model**

The embedding-based RNN model shows a clear improvement over the simple model. By introducing an embedding layer, the network learns dense vector representations for English words rather than treating them as isolated integer IDs. This allows the model to capture semantic similarities between words and provides richer input features for the recurrent layer.

The GRU layer with 64 units was chosen to balance model capacity and training efficiency. It is expressive enough to learn meaningful temporal patterns while still training quickly on the dataset. The learning rate of 0.001 continues to provide stable convergence with the Adam optimizer.

Training performance improves significantly across epochs, and the model reaches a much higher accuracy than the baseline. The predictions are still imperfect (some words repeat or appear in unnatural order) but the structure of the output is noticeably closer to valid French compared to the simple RNN. This demonstrates the value of word embeddings in machine translation pipelines.

Although this model is not yet capable of producing fluent translations, it represents a strong intermediate step.

### Model 3: Bidirectional RNNs (IMPLEMENTATION)
![RNN](images/bidirectional.png)
One restriction of a RNN is that it can't see the future input, only the past.  This is where bidirectional recurrent neural networks come in.  They are able to see the future data.

In [24]:
from tensorflow.keras.layers import Input, Bidirectional, GRU, TimeDistributed, Dense
from tensorflow.keras.models import Model

def bd_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a bidirectional RNN model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # Input: same 3D shape as in the simple_model -> (timesteps, features)
    input_seq = Input(shape=input_shape[1:])   # e.g. (21, 1)

    # Bidirectional GRU layer, returning a sequence (one vector per time step)
    bd_rnn = Bidirectional(GRU(64, return_sequences=True))(input_seq)

    # TimeDistributed Dense layer to get a probability distribution over
    # the French vocabulary for each time step
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(bd_rnn)

    # Build functional model
    model = Model(inputs=input_seq, outputs=logits)

    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model


tests.test_bd_model(bd_model)


# TODO: Train and Print prediction(s)

# Prepare input: same shape as for simple_model (batch, timesteps, 1)
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

# Build the bidirectional model
bd_rnn_model = bd_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

# Train the neural network
bd_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

# Print prediction(s)
print(logits_to_text(bd_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 6s 25ms/step - loss: 3.2172 - accuracy: 0.4602 - val_loss: nan - val_accuracy: 0.5165
Epoch 2/10
108/108 [==============================] - 2s 18ms/step - loss: 1.9600 - accuracy: 0.5533 - val_loss: nan - val_accuracy: 0.5830
Epoch 3/10
108/108 [==============================] - 2s 18ms/step - loss: 1.6167 - accuracy: 0.5940 - val_loss: nan - val_accuracy: 0.6079
Epoch 4/10
108/108 [==============================] - 2s 18ms/step - loss: 1.4698 - accuracy: 0.6155 - val_loss: nan - val_accuracy: 0.6215
Epoch 5/10
108/108 [==============================] - 2s 18ms/step - loss: 1.3770 - accuracy: 0.6271 - val_loss: nan - val_accuracy: 0.6327
Epoch 6/10
108/108 [==============================] - 2s 18ms/step - loss: 1.3073 - accuracy: 0.6411 - val_loss: nan - val_accuracy: 0.6497
Epoch 7/10
108/108 [==============================] - 2s 18ms/step - loss: 1.2532 - accuracy: 0.6524 - val_loss: nan - val_accuracy: 0.6560
Epoch 8/10
108/108 [

## **Conclusion for the Bidirectional RNN Model**

The bidirectional RNN model provides another improvement over the earlier architectures by allowing the network to process input sequences in both forward and backward directions. This gives the model access to more contextual information at each time step, which is especially useful for translation tasks where the meaning of a word often depends on both preceding and following words.

The model achieves steadily increasing accuracy during training and reaches a higher validation accuracy than the simple model, though still below the embedding-based model. This is expected, since the bidirectional GRU processes raw integer inputs rather than learned word embeddings. Even so, the bidirectional structure helps the network capture broader sentence-level patterns and reduces some of the repetition seen in previous models.

The predicted translation remains imperfect (certain words repeat and padding tokens dominate the end of the sequence) but the overall sentence structure is more coherent than in the baseline RNN. This demonstrates the benefit of bidirectional processing while also highlighting its limitations when used without embeddings or an encoder–decoder architecture.

Overall, the bidirectional RNN model forms a strong transitional step. It shows the value of richer sequence context.

### Model 4: Encoder-Decoder (IMPLEMENTATION)
Time to look at encoder-decoder models.  This model is made up of an encoder and decoder. The encoder creates a matrix representation of the sentence.  The decoder takes this matrix as input and predicts the translation as output.

Create an encoder-decoder model in the cell below.

### Model 5: Custom (IMPLEMENTATION)
Use everything you learned from the previous models to create a model that incorporates embedding and a bidirectional rnn into one model.

In [28]:
from tensorflow.keras.layers import Input, GRU, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.models import Model

def encdec_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train an encoder-decoder model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # Input: same 3D shape as in simple_model and bd_model -> (timesteps, features)
    encoder_input = Input(shape=input_shape[1:])   # e.g. (15, 1)

    # Encoder: GRU compresses the whole sequence into a context vector
    encoder_state = GRU(64, return_sequences=False)(encoder_input)

    # Repeat context vector for each output time step
    repeated_context = RepeatVector(output_sequence_length)(encoder_state)

    # Decoder: GRU generates an output sequence from the repeated context
    decoder_output = GRU(64, return_sequences=True)(repeated_context)

    # Output: probability distribution over French vocabulary at each time step
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(decoder_output)

    # Build and compile model
    model = Model(inputs=encoder_input, outputs=logits)

    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model


tests.test_encdec_model(encdec_model)


# TODO: Train and Print prediction(s)

# Prepare input with 3D shape (batch, timesteps, 1)
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

# Build encoder–decoder model
encdec_rnn_model = encdec_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

# Train the neural network
encdec_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

# Print prediction(s)
print(logits_to_text(encdec_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 6s 25ms/step - loss: 3.4954 - accuracy: 0.4124 - val_loss: nan - val_accuracy: 0.4445
Epoch 2/10
108/108 [==============================] - 2s 17ms/step - loss: 2.6016 - accuracy: 0.4674 - val_loss: nan - val_accuracy: 0.4824
Epoch 3/10
108/108 [==============================] - 2s 17ms/step - loss: 2.4299 - accuracy: 0.4897 - val_loss: nan - val_accuracy: 0.4997
Epoch 4/10
108/108 [==============================] - 2s 17ms/step - loss: 2.2983 - accuracy: 0.4977 - val_loss: nan - val_accuracy: 0.5000
Epoch 5/10
108/108 [==============================] - 2s 17ms/step - loss: 2.2147 - accuracy: 0.5007 - val_loss: nan - val_accuracy: 0.5024
Epoch 6/10
108/108 [==============================] - 2s 18ms/step - loss: 2.0818 - accuracy: 0.5108 - val_loss: nan - val_accuracy: 0.5175
Epoch 7/10
108/108 [==============================] - 2s 18ms/step - loss: 1.9255 - accuracy: 0.5304 - val_loss: nan - val_accuracy: 0.5433
Epoch 8/10
108/108 [

## Conclusion for the Encoder–Decoder Model

The encoder–decoder model represents a more advanced architecture designed specifically for sequence-to-sequence tasks such as machine translation. In this setup, the encoder compresses the entire input sentence into a single context vector, and the decoder reconstructs the output sequence from that fixed representation. This allows the model to learn a structured mapping between English and French sentences, even when the two languages differ in grammar and word order.


## Prediction (IMPLEMENTATION)

In [32]:
from tensorflow.keras.layers import Input, Embedding, Bidirectional, GRU, TimeDistributed, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Zorg dat je ergens in de notebook al een learning_rate hebt,
# maar voor de zekerheid zetten we hem hier ook:
learning_rate = 0.001


def model_final(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build a final model that uses embeddings and bidirectional RNNs.
    :param input_shape: Tuple of input shape (batch_size, timesteps)
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # Input is a sequence of integer word indices
    input_seq = Input(shape=(input_shape[1],))   # e.g. (max_french_sequence_length,)

    # Embedding layer to learn dense word representations
    embed = Embedding(
        input_dim=english_vocab_size,
        output_dim=128,
        input_length=input_shape[1]
    )(input_seq)

    # Bidirectional GRU to capture context from both directions
    rnn = Bidirectional(GRU(256, return_sequences=True))(embed)

    # Optional extra GRU layer for more capacity (still returns a sequence)
    rnn = GRU(256, return_sequences=True)(rnn)

    # TimeDistributed Dense layer to produce a probability distribution
    # over the French vocabulary at each time step
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(rnn)

    # Build and compile final model
    model = Model(inputs=input_seq, outputs=logits)
    model.compile(
        loss=sparse_categorical_crossentropy,
        optimizer=Adam(learning_rate),
        metrics=['accuracy']
    )

    return model


# Als je in de notebook een test hebt, kun je deze aanroepen:
# tests.test_model_final(model_final)


def final_predictions(x, y, x_tk, y_tk):
    """
    Gets predictions using the final model
    :param x: Preprocessed English data (integer sequences)
    :param y: Preprocessed French data (one-hot / sparse targets)
    :param x_tk: English tokenizer
    :param y_tk: French tokenizer
    """
    # 1) Pad English sequences to match the French sequence length
    output_sequence_length = y.shape[1]
    x_padded = pad(x, length=output_sequence_length)   # gebruikt jouw pad() functie

    # 2) shapes and vocab sizes
    input_shape = x_padded.shape
    english_vocab_size = len(x_tk.word_index) + 1
    french_vocab_size = len(y_tk.word_index) + 1

    # 3) train final model
    model = model_final(
        input_shape,
        output_sequence_length,
        english_vocab_size,
        french_vocab_size
    )

    model.fit(
        x_padded,
        y,
        batch_size=1024,
        epochs=10,
        validation_split=0.2
    )

    # ---------- DON'T EDIT ANYTHING BELOW THIS LINE ----------
    y_id_to_word = {value: key for key, value in y_tk.word_index.items()}
    y_id_to_word[0] = '<PAD>'

    sentence = 'he saw a old yellow truck'
    sentence = [x_tk.word_index[word] for word in sentence.split()]
    sentence = pad_sequences([sentence], maxlen=x_padded.shape[-1], padding='post')
    sentences = np.array([sentence[0], x_padded[0]])
    predictions = model.predict(sentences, len(sentences))

    print('Sample 1:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[0]]))
    print('Il a vu un vieux camion jaune')
    print('Sample 2:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[1]]))
    print(' '.join([y_id_to_word[np.max(x)] for x in y[0]]))


# Roep de functie aan met je eerder gepreprocessede data
final_predictions(preproc_english_sentences,
                  preproc_french_sentences,
                  english_tokenizer,
                  french_tokenizer)


Epoch 1/10
108/108 [==============================] - 17s 102ms/step - loss: 2.7273 - accuracy: 0.4720 - val_loss: 1.7682 - val_accuracy: 0.5827
Epoch 2/10
108/108 [==============================] - 10s 89ms/step - loss: 1.2656 - accuracy: 0.6847 - val_loss: 0.9429 - val_accuracy: 0.7578
Epoch 3/10
108/108 [==============================] - 9s 88ms/step - loss: 0.7430 - accuracy: 0.8054 - val_loss: 0.5935 - val_accuracy: 0.8387
Epoch 4/10
108/108 [==============================] - 9s 85ms/step - loss: 0.4911 - accuracy: 0.8617 - val_loss: 0.4241 - val_accuracy: 0.8769
Epoch 5/10
108/108 [==============================] - 9s 83ms/step - loss: 0.3660 - accuracy: 0.8924 - val_loss: 0.3237 - val_accuracy: 0.9038
Epoch 6/10
108/108 [==============================] - 9s 83ms/step - loss: 0.2909 - accuracy: 0.9129 - val_loss: 0.2631 - val_accuracy: 0.9213
Epoch 7/10
108/108 [==============================] - 9s 82ms/step - loss: 0.2339 - accuracy: 0.9300 - val_loss: 0.2107 - val_accuracy: 0.9

## Conclusion for the Final Model

The final model combines several ideas from the earlier experiments into a single, stronger architecture. It uses an embedding layer to learn dense representations of English words, followed by a stacked bidirectional GRU. The bidirectional layer lets the model see both past and future context, and the second GRU layer adds extra capacity to model complex patterns. A TimeDistributed Dense layer with softmax then predicts a French word at each time step.

Training curves show a steady improvement: the validation accuracy climbs from around 58% in the first epoch to about 96% by the tenth epoch, while the loss decreases smoothly. This indicates stable learning and a good fit to the dataset without clear signs of overfitting. The qualitative results confirm this: the model produces a perfect translation for the custom sentence (“he saw a old yellow truck”) and accurately reproduces a real example from the dataset, apart from trailing <PAD> tokens.

Compared with the earlier models (simple RNN, embedding-only, bidirectional, and encoder–decoder), the final model achieves much higher accuracy and far more natural translations. The combination of word embeddings, bidirectional recurrent layers, and increased model depth allows it to capture both local word relationships and long-range sentence structure. This final architecture therefore represents a strong end-to-end neural machine translation baseline.

## Submission
When you're ready to submit, complete the following steps:
1. Review the rubric to ensure your submission meets all requirements to pass
2. Generate an HTML version of this notebook

  - Run the next cell to attempt automatic generation (this is the recommended method in Workspaces)
  - Navigate to **FILE -> Download as -> HTML (.html)**
  - Manually generate a copy using `nbconvert` from your shell terminal
```
$ pip install nbconvert
$ python -m nbconvert machine_translation.ipynb
```
  
3. Submit the project

  - If you are in a Workspace, simply click the "Submit Project" button (bottom towards the right)
  
  - Otherwise, add the following files into a zip archive and submit them 
  - `helper.py`
  - `machine_translation.ipynb`
  - `machine_translation.html`
    - You can export the notebook by navigating to **File -> Download as -> HTML (.html)**.

### Generate the html

**Save your notebook before running the next cell to generate the HTML output.** Then submit your project.

In [ ]:
# Save before you run this cell!
!!jupyter nbconvert *.ipynb

## Optional Enhancements

This project focuses on learning various network architectures for machine translation, but we don't evaluate the models according to best practices by splitting the data into separate test & training sets -- so the model accuracy is overstated. Use the [`sklearn.model_selection.train_test_split()`](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function to create separate training & test datasets, then retrain each of the models using only the training set and evaluate the prediction accuracy using the hold out test set. Does the "best" model change?